# SC4002 - Lab Assigment

## Import and Configs

In [1]:
import os
import re
import math
import copy
import numpy as np
import pandas as pd
import random
import time

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import gensim.downloader as api

import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [2]:
DATA_FOLDER = r"data"
MODEL_FOLDER = r"model"

os.makedirs(DATA_FOLDER, exist_ok=True)
os.makedirs(MODEL_FOLDER, exist_ok=True)

DATASET_FILE = "en-US.jsonl"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Fixing seed to remove randomness, for better comparisons
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [4]:
RUN_PRETRAINING = False # Run MLM pretraining; if False, reuse the backbone checkpoints in MODEL_FOLDER
USE_MLM = False #Intent classifier uses the MLM-pretrained backbone; if False, the untrained one
RUN_INTENT_TRAINING = True # Train the intent classifier; if False, load its saved checkpoint

# Train the slot filler (not implemented yet)
RUN_SLOT_TRAINING = True

In [5]:
UNTRAINED_BACKBONE_PATH = os.path.join(MODEL_FOLDER, "untrained_backbone.pt")
BEST_BACKBONE_PATH = os.path.join(MODEL_FOLDER, "best_backbone.pt")
# Separate intent checkpoints so MLM and no-MLM runs do not overwrite each other
BEST_INTENT_PATH = os.path.join(
    MODEL_FOLDER, "best_intent_mlm.pt" if USE_MLM else "best_intent_no_mlm.pt"
)

## Data Loading

In [6]:
df = pd.read_json(os.path.join(DATA_FOLDER,DATASET_FILE), lines=True)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst example:")
print(df.iloc[0])

Dataset shape: (16521, 8)

Columns:
['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id']

First example:
id                                                           0
locale                                                   en-US
partition                                                 test
scenario                                                 alarm
intent                                               alarm_set
utt                            wake me up at five am this week
annot_utt    wake me up at [time : five am] [date : this week]
worker_id                                                    1
Name: 0, dtype: object


In [7]:
# Check the number of examples in each partition
print("Partition distribution:")
print(df["partition"].value_counts())

# Verify the official dataset split sizes
assert (df["partition"] == "train").sum() == 11514
assert (df["partition"] == "dev").sum() == 2033
assert (df["partition"] == "test").sum() == 2974

# Check the number of unique intents
print("\nNumber of unique intents:")
print(df["intent"].nunique())

# Check the number of examples per intent
print("\nExamples per intent:")
display(df["intent"].value_counts().describe())

# Check for missing values
print("\nMissing values:")
print(df[["utt", "annot_utt", "intent", "partition"]].isnull().sum())

# Check whether the dataset contains only en-US
print("\nLocales:")
print(df["locale"].value_counts())

Partition distribution:
partition
train    11514
test      2974
dev       2033
Name: count, dtype: int64

Number of unique intents:
60

Examples per intent:


count      60.000000
mean      275.350000
std       253.817352
min         6.000000
25%       114.750000
50%       183.000000
75%       315.500000
max      1150.000000
Name: count, dtype: float64


Missing values:
utt          0
annot_utt    0
intent       0
partition    0
dtype: int64

Locales:
locale
en-US    16521
Name: count, dtype: int64


In [8]:
train_df = df[df["partition"] == "train"].copy()
val_df = df[df["partition"] == "dev"].copy()
test_df = df[df["partition"] == "test"].copy()

In [9]:
x_train = train_df["utt"].tolist()
y_train = train_df["intent"].tolist()
z_train = train_df["annot_utt"].tolist()

x_val = val_df["utt"].tolist()
y_val = val_df["intent"].tolist()
z_val = val_df["annot_utt"].tolist()

x_test = test_df["utt"].tolist()
y_test = test_df["intent"].tolist()
z_test = test_df["annot_utt"].tolist()

In [10]:
print("Training examples:", len(x_train))
print("Validation examples:", len(x_val))
print("Test examples:", len(x_test))

print("\nExample utterance:", x_train[0])
print("Example intent:", y_train[0])
print("Example annotated utterance:", z_train[0])

Training examples: 11514
Validation examples: 2033
Test examples: 2974

Example utterance: wake me up at nine am on friday
Example intent: alarm_set
Example annotated utterance: wake me up at [time : nine am] on [date : friday]


In [11]:
glove = api.load("glove-wiki-gigaword-100")

In [12]:
# Check vocabulary size
print("Vocabulary size:", len(glove.index_to_key))

# Check embedding dimension
print("Embedding dimension:", glove.vector_size)

# Get the vector for a word
print("Vector for 'music':")
print(glove["music"])

Vocabulary size: 400000
Embedding dimension: 100
Vector for 'music':
[ 1.2705e-01  3.7253e-01  2.3076e-01 -1.0727e-01  1.4273e+00  7.7990e-01
 -5.5009e-02 -1.8909e-01 -4.2828e-01  2.8443e-01  1.1920e-01 -6.1978e-01
 -3.9817e-01  1.9127e-01 -3.2098e-01 -2.4116e-01  6.0927e-01 -3.5765e-01
  2.0159e-01  9.1107e-01 -4.8977e-01 -2.8970e-01  1.0098e+00 -6.6149e-01
  7.0414e-02  1.3353e-01  4.2234e-01 -8.0188e-02  1.0722e+00  2.6426e-01
 -7.3982e-01  6.4958e-01 -4.8367e-01  8.5837e-01 -3.9008e-01 -1.3020e-01
  2.2331e-01  2.0818e-01 -4.7260e-01 -1.7275e+00  3.7209e-01  5.5544e-01
 -8.9025e-01  8.7442e-01 -2.9063e-01 -7.1175e-01 -3.3437e-01 -4.1141e-01
  9.9049e-01 -4.6207e-01 -2.5777e-01  4.9513e-02 -2.2242e-01 -3.3757e-01
 -1.5677e-01 -2.9576e+00 -5.8878e-01  2.2432e-01  1.6375e+00  1.0263e+00
  8.9573e-01  1.3718e+00 -3.7709e-01 -8.2095e-02  1.6339e-01 -2.5294e-01
  1.0987e+00 -2.0032e-03  7.9235e-01 -2.9403e-01 -2.6906e-01  2.3123e-01
  3.3826e-01  1.7150e-01 -3.5479e-01 -1.3497e-01  6.441

In [13]:
token_pattern = re.compile(r'\w+|[^\w\s]')

def tokenize_with_offsets(text):
    matches = token_pattern.finditer(text)
    return [(m.group().lower(), m.start(), m.end()) for m in matches]

def tokenize(text):
    return [token for token, _, _ in tokenize_with_offsets(text)]

In [14]:
def check_glove_coverage(texts, glove):
    all_tokens = [
        token
        for text in texts
        for token in tokenize(text)
    ]

    unique_tokens = set(all_tokens)
    known_tokens = {token for token in unique_tokens if token in glove}

    total_tokens = len(all_tokens)
    total_unique = len(unique_tokens)
    known_count = sum(token in glove for token in all_tokens)

    print(f"Total tokens: {total_tokens:,}")
    print(f"Unique words: {total_unique:,}")
    print(f"Unique words covered: {len(known_tokens):,}")
    print(f"Unique words OOV: {total_unique - len(known_tokens):,}")

    print(f"\nUnique-word coverage: {len(known_tokens) / total_unique * 100:.2f}%")
    print(f"Token coverage: {known_count / total_tokens * 100:.2f}%")

    oov_words = sorted(unique_tokens - known_tokens)
    print("\nFirst 30 OOV words:")
    print(oov_words[:30])

In [15]:
# Check each split
print("TRAINING SET")
check_glove_coverage(x_train, glove)

print("\nVALIDATION SET")
check_glove_coverage(x_val, glove)

print("\nTEST SET")
check_glove_coverage(x_test, glove)

TRAINING SET
Total tokens: 83,008
Unique words: 5,144
Unique words covered: 4,963
Unique words OOV: 181

Unique-word coverage: 96.48%
Token coverage: 99.67%

First 30 OOV words:
['altucher', 'apahelp', 'apig', 'applebees', 'badpho', 'bardonia', 'bbcnews', 'beanos', 'beiber', 'beibers', 'bestbuy', 'blahblah', 'bobburgers', 'brexit', 'brightener', 'btaxes', 'burmuda', 'calone', 'capabara', 'capuccino', 'channin', 'chhana', 'cnncom', 'comcastcom', 'comcasts', 'companyname', 'dalls', 'dancom', 'dbell', 'ddlj']

VALIDATION SET
Total tokens: 14,522
Unique words: 2,088
Unique words covered: 2,052
Unique words OOV: 36

Unique-word coverage: 98.28%
Token coverage: 99.70%

First 30 OOV words:
['alzacz', 'arrahman', 'ashmit', 'bakey', 'brexit', 'burmuda', 'capricorns', 'culotte', 'demonetisation', 'dooa', 'eggroll', 'enalen', 'erosser', 'finlee', 'gavenda', 'gbaby', 'harrypotter', 'iliza', 'metion', 'obaba', 'orevena', 'palhunik', 'phillyx', 'reveca', 'rihana', 'schleshinger', 'swatishs', 'tiangl

In [16]:
# Special token IDs
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
MASK_TOKEN = "<MASK>"

PAD_ID = 0
UNK_ID = 1
MASK_ID = 2

# Tokenize the training utterances
tokenized_x_train = [tokenize(text) for text in x_train]

# Build vocabulary using the training set
vocab_words = sorted({
    token
    for tokens in tokenized_x_train
    for token in tokens
})

# Create word-to-ID mapping
word_to_id = {
    PAD_TOKEN: PAD_ID,
    UNK_TOKEN: UNK_ID,
    MASK_TOKEN: MASK_ID
}

for word in vocab_words:
    if word not in word_to_id:
        word_to_id[word] = len(word_to_id)

# Create reverse mapping
id_to_word = {idx: word for word, idx in word_to_id.items()}

print("Training vocabulary size (incl. PAD/UNK/MASK):", len(word_to_id))
print("Training tokens missing from GloVe:", sum(word not in glove for word in vocab_words))

Training vocabulary size (incl. PAD/UNK/MASK): 5147
Training tokens missing from GloVe: 181


In [17]:
USE_LEMMA_FALLBACK = False

lemmatizer = WordNetLemmatizer()

def lemmatize_token(token):
    # Try the noun form first (plurals), then the verb form (tenses)
    lemma = lemmatizer.lemmatize(token, pos='n')
    if lemma == token:
        lemma = lemmatizer.lemmatize(token, pos='v')
    return lemma

def token_to_id(token, word_to_id, use_lemma):
    # Known words keep their own ID; only unseen words fall back to their lemma
    if token in word_to_id:
        return word_to_id[token]
    if use_lemma:
        lemma = lemmatize_token(token)
        if lemma in word_to_id:
            return word_to_id[lemma]
    return UNK_ID

In [18]:
embedding_dim = glove.vector_size
vocab_size = len(word_to_id)

rng = np.random.default_rng(SEED)

embedding_matrix = rng.normal(
    0, 0.1, (vocab_size, embedding_dim)
).astype(np.float32)

embedding_matrix[PAD_ID] = 0

for word, idx in word_to_id.items():
    if word in glove:
        embedding_matrix[idx] = glove[word]

print("Embedding matrix shape:", embedding_matrix.shape)
print("Padding vector:", embedding_matrix[PAD_ID])
print("Unknown vector shape:", embedding_matrix[UNK_ID].shape)
print("Words missing from GloVe:",
      sum(word not in glove for word in vocab_words))


Embedding matrix shape: (5147, 100)
Padding vector: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]
Unknown vector shape: (100,)
Words missing from GloVe: 181


In [19]:
max_train_len = max(len(tokens) for tokens in tokenized_x_train)
max_len = max(64, max_train_len)
print("Longest training sentence:", max_train_len)
print("Padding length:", max_len)

Longest training sentence: 35
Padding length: 64


In [20]:
intents = sorted(set(y_train))

intent_to_id = {
    intent: idx
    for idx, intent in enumerate(intents)
}

id_to_intent = {
    idx: intent
    for intent, idx in intent_to_id.items()
}

num_classes = len(intent_to_id)

print("Number of intent classes:", num_classes)
print("\nExample mappings:")
for intent in intents[:5]:
    print(f"{intent} -> {intent_to_id[intent]}")

Number of intent classes: 60

Example mappings:
alarm_query -> 0
alarm_remove -> 1
alarm_set -> 2
audio_volume_down -> 3
audio_volume_mute -> 4


In [21]:
slot_pattern = re.compile(r'\[\s*([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]')
SLOT_PAD_ID = -100  

def parse_annotated_utterance(utt, annot_utt):
    spans = []
    plain_text = ''
    last_end = 0

    for match in slot_pattern.finditer(annot_utt):
        plain_text += annot_utt[last_end:match.start()]
        slot_type = match.group(1).strip()
        slot_value = match.group(2)
        start = len(plain_text)
        plain_text += slot_value
        spans.append({
            'type': slot_type,
            'start': start,
            'end': len(plain_text),
            'text': slot_value
        })
        last_end = match.end()

    plain_text += annot_utt[last_end:]
    if plain_text != utt:
        raise ValueError(f"Slot annotation does not match utterance: {utt!r}")

    return spans


In [22]:
def make_aligned_tokens(utt, spans):
    tags = []
    offsets = []
    prev_slot = None

    for word, start, end in tokenize_with_offsets(utt):
        current_slot = None
        for i, span in enumerate(spans):
            if start < span['end'] and end > span['start']:
                if start < span['start'] or end > span['end']:
                    raise ValueError(f"A token cuts across a slot boundary: {utt!r}")
                if current_slot is not None:
                    raise ValueError(f"A token overlaps two slots: {utt!r}")
                current_slot = i

        if current_slot is None:
            tags.append('O')
        else:
            prefix = 'I' if current_slot == prev_slot else 'B'
            tags.append(f"{prefix}-{spans[current_slot]['type']}")

        offsets.append((start, end))
        prev_slot = current_slot

    for span in spans:
        if not any(s >= span['start'] and e <= span['end'] for s, e in offsets):
            raise ValueError(f"Could not align slot {span['text']!r} in {utt!r}")

    return tags, offsets


In [23]:
def prepare_joint_records(split_df):
    records = []
    
    for row_index, row in split_df.iterrows():
        text = row['utt']
        spans = parse_annotated_utterance(text, row['annot_utt'])
        tags, offsets = make_aligned_tokens(text, spans)
        tokens = tokenize(text)

        if len(tokens) != len(tags):
            raise ValueError(f"Token and slot lengths differ at row {row_index}")

        records.append({
            'source_row': int(row_index),
            'utt': text,
            'tokens': tokens,
            'offsets': offsets,
            'gold_spans': spans,
            'slot_tags': tags,
            'intent': row['intent']
        })
    return records


In [24]:
train_records = prepare_joint_records(train_df)
val_records = prepare_joint_records(val_df)
test_records = prepare_joint_records(test_df)

training_slot_types = sorted({span['type'] for row in train_records for span in row['gold_spans']})
slot_tags = ['O']

for slot_type in training_slot_types:
    slot_tags.extend([f'B-{slot_type}', f'I-{slot_type}'])

slot_to_id = {tag: i for i, tag in enumerate(slot_tags)}
id_to_slot = {i: tag for tag, i in slot_to_id.items()}
num_slot_labels = len(slot_to_id)

print('Joint dataset sizes:', len(train_records), len(val_records), len(test_records))
print('Slot types in training:', len(training_slot_types))
print('BIO labels:', num_slot_labels)
print('Example:', train_records[0]['slot_tags'])

Joint dataset sizes: 11514 2033 2974
Slot types in training: 55
BIO labels: 111
Example: ['O', 'O', 'O', 'O', 'B-time', 'I-time', 'O', 'B-date']


In [25]:
class JointNLUDataset(Dataset):
    def __init__(self, records, word_to_id, intent_to_id, 
                 slot_to_id, max_len, use_lemma=False):
        self.records = records
        self.input_ids = []
        self.attention_masks = []
        self.intent_ids = []
        self.slot_ids = []

        for row in records:
            words = [token_to_id(word, word_to_id, use_lemma) for word in row['tokens']]
            slots = [slot_to_id[tag] for tag in row['slot_tags']]
            if len(words) != len(slots):
                raise ValueError('Token and slot label counts do not match')
            if len(words) > max_len:
                raise ValueError(f"Sentence has {len(words)} tokens, limit is {max_len}")

            padding = max_len - len(words)
            self.input_ids.append(torch.tensor(words + [PAD_ID] * padding))
            self.attention_masks.append(torch.tensor([1] * len(words) + [0] * padding, 
                                                     dtype=torch.bool))
            self.intent_ids.append(torch.tensor(intent_to_id[row['intent']]))
            self.slot_ids.append(torch.tensor(slots + [SLOT_PAD_ID] * padding))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        return {
            'input_ids': self.input_ids[index],
            'attention_mask': self.attention_masks[index],
            'intent_labels': self.intent_ids[index],
            'slot_labels': self.slot_ids[index],
            'utt': row['utt'],
            'offsets': row['offsets'],
            'gold_spans': row['gold_spans'],
            'source_row': row['source_row']
        }

In [26]:
def joint_collate_fn(samples):
    batch = {}
    for key in ('input_ids', 'attention_mask', 'intent_labels', 'slot_labels'):
        batch[key] = torch.stack([item[key] for item in samples])
    for key in ('utt', 'offsets', 'gold_spans', 'source_row'):
        batch[key] = [item[key] for item in samples]
    return batch

In [27]:
train_joint_dataset = JointNLUDataset(train_records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=USE_LEMMA_FALLBACK)
val_joint_dataset = JointNLUDataset(val_records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=USE_LEMMA_FALLBACK)
test_joint_dataset = JointNLUDataset(test_records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=USE_LEMMA_FALLBACK)

In [28]:
batch_size = 64

train_joint_loader = DataLoader(train_joint_dataset, 
                                batch_size=batch_size, 
                                shuffle=True,
                                collate_fn=joint_collate_fn,
                                generator=torch.Generator().manual_seed(SEED))

val_joint_loader = DataLoader(val_joint_dataset, 
                              batch_size=batch_size, 
                              shuffle=False,
                              collate_fn=joint_collate_fn)

test_joint_loader = DataLoader(test_joint_dataset, 
                               batch_size=batch_size, 
                               shuffle=False,
                               collate_fn=joint_collate_fn)

batch = next(iter(train_joint_loader))
print('Input IDs:', batch['input_ids'].shape)
print('Intent labels:', batch['intent_labels'].shape)
print('Slot labels:', batch['slot_labels'].shape)
print('Attention mask:', batch['attention_mask'].shape)

assert batch['input_ids'].shape == batch['slot_labels'].shape
assert torch.equal(batch['slot_labels'] == SLOT_PAD_ID, ~batch['attention_mask'])

Input IDs: torch.Size([64, 64])
Intent labels: torch.Size([64])
Slot labels: torch.Size([64, 64])
Attention mask: torch.Size([64, 64])


In [29]:
def build_vocab(records):
    # Vocabulary from the given training records only
    vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID, MASK_TOKEN: MASK_ID}
    for word in sorted({t for row in records for t in row['tokens']}):
        vocab[word] = len(vocab)
    return vocab

def build_embedding_matrix(vocab):
    matrix_rng = np.random.default_rng(SEED)
    matrix = matrix_rng.normal(0, 0.1, (len(vocab), embedding_dim)).astype(np.float32)
    matrix[PAD_ID] = 0
    for word, idx in vocab.items():
        if word in glove:
            matrix[idx] = glove[word]
    return matrix

def make_loader(records, vocab, use_lemma=False, shuffle=False):
    dataset = JointNLUDataset(records, vocab, intent_to_id, slot_to_id, max_len, use_lemma=use_lemma)
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=joint_collate_fn, generator=generator)

def lemmatize_records(records):
    # Lemmatize model input tokens only; utt, offsets and gold spans stay original
    return [{**row, 'tokens': [lemmatize_token(t) for t in row['tokens']]} for row in records]

# The helpers must reproduce the main pipeline exactly
assert build_vocab(train_records) == word_to_id
assert np.array_equal(build_embedding_matrix(word_to_id), embedding_matrix)
print("Helpers reproduce the main vocabulary and embedding matrix")

Helpers reproduce the main vocabulary and embedding matrix


## Model Architecture

In [30]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        # x shape: (batch_size, sequence_length, d_model)
        return x + self.pe[:, :x.size(1), :]

In [31]:
class BackboneEncoder(nn.Module):
    def __init__(
        self,
        embedding_matrix,
        pad_id,
        max_len=128,
        num_layers=4,
        num_heads=4,
        dim_feedforward=256,
        dropout=0.1,
    ):
        super().__init__()

        embedding_matrix = torch.as_tensor(
            embedding_matrix,
            dtype=torch.float32,
        )

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=True,
            padding_idx=pad_id,
        )

        d_model = embedding_matrix.size(1)

        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            max_len=max_len,
        )

        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
        )

        self.pad_id = pad_id
        self.d_model = d_model

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)
        x = self.positional_encoding(x)
        x = self.dropout(x)

        padding_mask = ~attention_mask.bool()

        x = self.transformer_encoder(
            x,
            src_key_padding_mask=padding_mask,
        )

        return x

In [32]:
class BackboneReconstructor(nn.Module):
    def __init__(
        self,
        d_model,
        vocab_size,
        num_layers=2,
        num_heads=4,
        dim_feedforward=256,
        dropout=0.1,
    ):
        super().__init__()

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            decoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
        )

        self.output_projection = nn.Linear(
            d_model,
            vocab_size,
        )

    def forward(self, hidden_states, attention_mask):
        padding_mask = ~attention_mask.bool()
        x = self.transformer(
            hidden_states,
            src_key_padding_mask=padding_mask,
        )
        logits = self.output_projection(x)

        return logits

In [33]:
class Backbone(nn.Module):
    def __init__(self, encoder, reconstructor):
        super().__init__()

        self.encoder = encoder
        self.reconstructor = reconstructor

    def forward(self, input_ids, attention_mask, labels=None):
        hidden_states = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        logits = self.reconstructor(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
        )

        loss = None

        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            )

        return {
            "loss": loss,
            "logits": logits,
        }

In [34]:
class IntentClassifier(nn.Module):
    def __init__(
        self,
        backbone_encoder,
        num_classes,
        num_layers=2,
        nhead=4,
        dim_feedforward=256,
        dropout=0.1,
        pad_id=PAD_ID,
        use_backbone_layers=True,
    ):
        super().__init__()

        # Reuse the existing encoder; do not copy it.
        self.backbone_encoder = backbone_encoder
        self.pad_id = pad_id
        # If False, only the frozen embedding + positional encoding are used
        self.use_backbone_layers = use_backbone_layers

        # Freeze all backbone parameters.
        for param in self.backbone_encoder.parameters():
            param.requires_grad = False

        d_model = self.backbone_encoder.d_model

        if d_model % nhead != 0:
            raise ValueError(
                f"d_model={d_model} must be divisible by nhead={nhead}"
            )

        # Additional Transformer trained for intent classification.
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.intent_encoder = nn.TransformerEncoder(
            layer,
            num_layers=num_layers,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask):
        padding_mask = ~attention_mask.bool()

        # Keep the frozen backbone deterministic, including its dropout.
        with torch.no_grad():
            if self.use_backbone_layers:
                hidden_states = self.backbone_encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )
            else:
                # GloVe + positional encoding only, as in the old IntentTransformer
                hidden_states = self.backbone_encoder.positional_encoding(
                    self.backbone_encoder.embedding(input_ids)
                )

        # The intent-specific Transformer is trainable.
        hidden_states = self.intent_encoder(
            hidden_states,
            src_key_padding_mask=padding_mask,
        )

        # Masked mean pooling.
        valid_tokens = attention_mask.unsqueeze(-1).to(
            hidden_states.dtype
        )

        pooled = (hidden_states * valid_tokens).sum(dim=1)
        pooled = pooled / valid_tokens.sum(dim=1).clamp(min=1.0)

        logits = self.classifier(self.dropout(pooled))

        return logits

## Pretraining Loop

In [35]:
encoder = BackboneEncoder(
    embedding_matrix=embedding_matrix,
    pad_id=PAD_ID,
    max_len=128,
    num_layers=5,
    num_heads=4,
    dim_feedforward=1024,
    dropout=0.1,
)

reconstructor = BackboneReconstructor(
    d_model=encoder.d_model,
    vocab_size=embedding_matrix.shape[0],
    num_layers=2,
    num_heads=4,
    dim_feedforward=256,
    dropout=0.1,
)

C:\Users\iamgr\AppData\Local\Temp\ipykernel_22792\1509289189.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(
C:\Users\iamgr\AppData\Local\Temp\ipykernel_22792\1812997011.py:23: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


In [36]:
mlm_model = Backbone(
    encoder=encoder,
    reconstructor=reconstructor,
)

In [37]:
MAX_EPOCH = 500
early_stop_patience = 10
best_loss = float("inf")
epochs_without_improvement = 0

train_loss_history = []

In [38]:
def create_mlm_inputs(input_ids, attention_mask, mask_token_id=MASK_ID, mlm_probability=0.15):
    mlm_input_ids = input_ids.clone()
    mlm_labels = input_ids.clone()

    # Only valid, non-padding tokens can be selected
    special_mask = input_ids.eq(PAD_ID)
    eligible_tokens = attention_mask.bool() & ~special_mask

    probability_matrix = torch.full(
        input_ids.shape,
        mlm_probability,
        device=input_ids.device,
    )

    masked_positions = (
        torch.bernoulli(probability_matrix).bool()
        & eligible_tokens
    )

    # Compute loss only at masked positions
    mlm_labels[~masked_positions] = -100

    # Replace selected tokens with the mask token
    mlm_input_ids[masked_positions] = mask_token_id

    return mlm_input_ids, mlm_labels

In [39]:
mlm_model = mlm_model.to(device)

optimizer = optim.AdamW(
    (p for p in mlm_model.parameters() if p.requires_grad),
    lr=1e-3,
    weight_decay=0.01,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=3,
    min_lr=1e-6,
)

In [40]:
def train_mlm_epoch(
    model,
    dataloader,
    optimizer,
    device,
    mask_token_id,
):
    model.train()

    total_loss = 0.0
    total_batches = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        mlm_input_ids, mlm_labels = create_mlm_inputs(
            input_ids=input_ids,
            attention_mask=attention_mask,
            mask_token_id=mask_token_id,
        )

        # Skip a batch if random masking selected no tokens
        if not (mlm_labels != -100).any():
            continue

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=mlm_input_ids,
            attention_mask=attention_mask,
            labels=mlm_labels,
        )

        loss = outputs["loss"]
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    if total_batches == 0:
        raise RuntimeError(
            "No masked tokens were selected in any batch."
        )

    return total_loss / total_batches

In [41]:
if RUN_PRETRAINING:
    torch.save(
        {
            k: v.detach().cpu().clone()
            for k, v in mlm_model.encoder.state_dict().items()
        },
        UNTRAINED_BACKBONE_PATH,
    )

    for epoch in range(MAX_EPOCH):
        train_loss = train_mlm_epoch(
            model=mlm_model,
            dataloader=train_joint_loader,
            optimizer=optimizer,
            device=device,
            mask_token_id=MASK_ID,
        )

        train_loss_history.append(train_loss)

        # Update learning rate based on training loss
        scheduler.step(train_loss)

        # Early stopping check
        if train_loss < best_loss:
            best_loss = train_loss
            epochs_without_improvement = 0

            # Save the best model
            best_model_state = {
                k: v.detach().cpu().clone()
                for k, v in mlm_model.state_dict().items()
            }

            torch.save(
                best_model_state,
                BEST_BACKBONE_PATH,
            )
        else:
            epochs_without_improvement += 1

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch [{epoch + 1}/{MAX_EPOCH}] "
            f"| Train MLM Loss: {train_loss:.4f} "
            f"| Best Loss: {best_loss:.4f} "
            f"| LR: {current_lr:.2e}"
        )

        if epochs_without_improvement >= early_stop_patience:
            print(f"Early stopping triggered at epoch {epoch + 1}.")
            break

    # Restore the best model weights
    mlm_model.load_state_dict(best_model_state)

    print(f"Best MLM Loss: {best_loss:.4f}")
else:
    print("Skipping MLM pretraining.")

    # Keep the no-MLM baseline available even if pretraining never ran
    if not os.path.exists(UNTRAINED_BACKBONE_PATH):
        torch.save(
            {
                k: v.detach().cpu().clone()
                for k, v in mlm_model.encoder.state_dict().items()
            },
            UNTRAINED_BACKBONE_PATH,
        )

Skipping MLM pretraining.


## Training Loop (Intent Classifier)

In [42]:
# Load either the best MLM checkpoint or the untrained backbone.
backbone_path = BEST_BACKBONE_PATH if USE_MLM else UNTRAINED_BACKBONE_PATH

if not os.path.exists(backbone_path):
    raise FileNotFoundError(
        f"{backbone_path} not found. Set RUN_PRETRAINING = True to create it."
    )

checkpoint = torch.load(
    backbone_path,
    map_location=device,
    weights_only=True,
)

if USE_MLM:
    # The MLM checkpoint holds the full Backbone (encoder + reconstructor)
    mlm_model.load_state_dict(checkpoint)
    print("Pretrained MLM backbone loaded.")
else:
    # The untrained checkpoint holds the encoder only
    mlm_model.encoder.load_state_dict(checkpoint)
    print("Untrained backbone loaded (no MLM).")

Untrained backbone loaded (no MLM).


In [43]:
# Same seed as the old make_model so the head starts from the same weights
torch.manual_seed(SEED)

intent_model = IntentClassifier(
    backbone_encoder=mlm_model.encoder,
    num_classes=num_classes,
    num_layers=4,
    nhead=4,
    dim_feedforward=512,
    dropout=0.3,
    pad_id=PAD_ID,
    use_backbone_layers=USE_MLM,
).to(device)

In [44]:
# Freeze backbone behavior, including dropout.
intent_model.backbone_encoder.eval()

criterion = nn.CrossEntropyLoss()

intent_optimizer = torch.optim.AdamW(
    [
        *intent_model.intent_encoder.parameters(),
        *intent_model.classifier.parameters(),
    ],
    lr=1e-3,
    weight_decay=1e-4,
)

# Reduce learning rate when validation accuracy stops improving.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    intent_optimizer,
    mode="max",
    factor=0.1,
    patience=10,
    min_lr=1e-6,
)

# Early stopping settings.
PATIENCE = 20
best_val_acc = 0.0
epochs_without_improvement = 0

In [45]:
def train_intent_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device,
):
    model.train()
    model.backbone_encoder.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch in train_loader:
        x_batch = batch["input_ids"].to(device)
        mask_batch = batch["attention_mask"].to(device)
        y_batch = batch["intent_labels"].to(device).long()

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            input_ids=x_batch,
            attention_mask=mask_batch,
        )

        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += batch_size

    avg_loss = total_loss / max(total, 1)
    accuracy = correct / max(total, 1)

    return avg_loss, accuracy

In [46]:
@torch.no_grad()
def evaluate_intent(
    model,
    dataloader,
    criterion,
    device,
):
    model.eval()

    # Keep the pretrained backbone in evaluation mode.
    model.backbone_encoder.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["intent_labels"].to(device).long()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += batch_size

    return {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
    }

In [47]:
if RUN_INTENT_TRAINING:
    best_val_acc = 0.0

    for epoch in range(MAX_EPOCH):
        train_loss, train_acc = train_intent_epoch(
            intent_model,
            train_joint_loader,
            criterion,
            intent_optimizer,
            device,
        )

        val_metrics = evaluate_intent(
            intent_model,
            val_joint_loader,
            criterion,
            device,
        )

        val_loss = val_metrics["loss"]
        val_acc = val_metrics["accuracy"]

        # Update learning rate based on validation accuracy.
        old_lr = intent_optimizer.param_groups[0]["lr"]
        scheduler.step(val_acc)
        new_lr = intent_optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch + 1}/{MAX_EPOCH} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"LR: {new_lr:.2e}"
        )

        if new_lr < old_lr:
            print(f"Learning rate reduced: {old_lr:.2e} -> {new_lr:.2e}")

        # Save the best model and reset early stopping counter.
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0

            torch.save(
                intent_model.state_dict(),
                BEST_INTENT_PATH,
            )

            print("Saved best intent classifier.")

        else:
            epochs_without_improvement += 1
            print(
                f"No validation accuracy improvement: "
                f"{epochs_without_improvement}/{PATIENCE}"
            )

        # Early stopping.
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break
else:
    print("Skipping intent classifier training.")

C:\Users\iamgr\Desktop\SC4002-Group-Assignment\venv\Lib\site-packages\torch\nn\modules\transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(


Epoch 1/500 | Train Loss: 2.3835 | Train Acc: 0.4050 | Val Loss: 1.3152 | Val Acc: 0.6454 | LR: 1.00e-03
Saved best intent classifier.
Epoch 2/500 | Train Loss: 1.2472 | Train Acc: 0.6654 | Val Loss: 0.9362 | Val Acc: 0.7447 | LR: 1.00e-03
Saved best intent classifier.
Epoch 3/500 | Train Loss: 0.9639 | Train Acc: 0.7347 | Val Loss: 0.8201 | Val Acc: 0.7796 | LR: 1.00e-03
Saved best intent classifier.
Epoch 4/500 | Train Loss: 0.8261 | Train Acc: 0.7727 | Val Loss: 0.7699 | Val Acc: 0.8003 | LR: 1.00e-03
Saved best intent classifier.
Epoch 5/500 | Train Loss: 0.7463 | Train Acc: 0.7909 | Val Loss: 0.7582 | Val Acc: 0.7959 | LR: 1.00e-03
No validation accuracy improvement: 1/20
Epoch 6/500 | Train Loss: 0.6669 | Train Acc: 0.8160 | Val Loss: 0.7358 | Val Acc: 0.8126 | LR: 1.00e-03
Saved best intent classifier.
Epoch 7/500 | Train Loss: 0.6172 | Train Acc: 0.8232 | Val Loss: 0.7146 | Val Acc: 0.8219 | LR: 1.00e-03
Saved best intent classifier.
Epoch 8/500 | Train Loss: 0.5542 | Train Acc

In [48]:
if not os.path.exists(BEST_INTENT_PATH):
    raise FileNotFoundError(
        f"{BEST_INTENT_PATH} not found. Set RUN_INTENT_TRAINING = True to create it."
    )

checkpoint = torch.load(
    BEST_INTENT_PATH,
    map_location=device,
    weights_only=True,
)

intent_model.load_state_dict(checkpoint)
intent_model.eval()

print("Best intent classifier loaded.")

Best intent classifier loaded.


In [49]:
eval_metrics = evaluate_intent(
        intent_model,
        val_joint_loader,
        criterion,
        device,
    )

print(f"Val Acc: {eval_metrics['accuracy']:.4f}")

eval_metrics = evaluate_intent(
        intent_model,
        test_joint_loader,
        criterion,
        device,
    )

print(f"Test Acc: {eval_metrics['accuracy']:.4f}")

Val Acc: 0.8554
Test Acc: 0.8393


## Training Loop (Slot Filling)

In [50]:
if RUN_SLOT_TRAINING:
    print("Slot filling training is not implemented yet; skipping.")
else:
    print("Skipping slot filling training.")

Slot filling training is not implemented yet; skipping.


## Old Intent Classification + Lemmatization Experiments

In [51]:
MODEL_CONFIG = dict(nhead=4, num_layers=4, dim_feedforward=512, dropout=0.3)

def make_model(embedding_matrix):
    # Same seed for every variant so all models start from the same weights
    torch.manual_seed(SEED)
    return IntentTransformer(embedding_matrix=embedding_matrix, num_classes=num_classes,
                             pad_id=PAD_ID, **MODEL_CONFIG)

model = make_model(embedding_matrix)
print(model)

NameError: name 'IntentTransformer' is not defined

In [ ]:
model.eval()

with torch.no_grad():
    logits = model(batch['input_ids'])

print("Logits shape:", logits.shape)

In [ ]:
model = model.to(device)

# Loss function for multi-class classification
criterion = nn.CrossEntropyLoss()

print("Device:", device)
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
print("Total parameters:", sum(
    p.numel() for p in model.parameters()
))

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch in train_loader:
        x_batch = batch['input_ids'].to(device)
        y_batch = batch['intent_labels'].to(device)

        # Clear gradients from the previous batch
        optimizer.zero_grad()

        # Forward pass
        logits = model(x_batch)
        loss = criterion(logits, y_batch)

        # Backpropagation and parameter update
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        total_loss += loss.item() * x_batch.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in data_loader:
            x_batch = batch['input_ids'].to(device)
            y_batch = batch['intent_labels'].to(device)

            # Forward pass only
            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            # Track loss and accuracy
            total_loss += loss.item() * x_batch.size(0)
            predictions = logits.argmax(dim=1)
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
NUM_EPOCHS = 50

def run_experiment(embedding_matrix, train_loader, val_loaders, prefix='intent'):
    # Train one model; track the best epoch separately for each validation encoding
    exp_model = make_model(embedding_matrix).to(device)
    exp_optimizer = optim.AdamW(filter(lambda p: p.requires_grad, exp_model.parameters()),
                                lr=1e-3, weight_decay=1e-4)
    best = {label: {'acc': -1.0, 'epoch': None, 'state': None} for label in val_loaders}
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        start = time.time()
        train_loss, train_acc = train_one_epoch(exp_model, train_loader, criterion, exp_optimizer, device)
        row = {'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc}

        for label, loader in val_loaders.items():
            val_loss, val_acc = evaluate(exp_model, loader, criterion, device)
            row[f'{label} val_loss'] = val_loss
            row[f'{label} val_acc'] = val_acc
            if val_acc > best[label]['acc']:
                best[label] = {'acc': val_acc, 'epoch': epoch,
                               'state': copy.deepcopy(exp_model.state_dict())}

        row['seconds'] = time.time() - start
        history.append(row)
        val_text = " | ".join(f"{label} {row[f'{label} val_acc'] * 100:.2f}%" for label in val_loaders)
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
              f"{val_text} | {row['seconds']:.1f}s")

    results = {}
    for label, info in best.items():
        filename = f"{prefix}_{label.lower().replace(' ', '_')}_best.pt"
        torch.save(info['state'], filename)
        results[label] = {'Best epoch': info['epoch'],
                          'Best intent accuracy (%)': round(100 * info['acc'], 2),
                          'Checkpoint': filename}
        print(f"{label}: best val {info['acc'] * 100:.2f}% at epoch {info['epoch']} -> {filename}")
    return exp_model, history, results

all_results = {}

In [ ]:
# Training inputs are identical with and without the fallback, so one model serves both
train_with_lemma = JointNLUDataset(train_records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=True)
assert all(torch.equal(a, b) for a, b in zip(train_joint_dataset.input_ids, train_with_lemma.input_ids))
print("Verified: training token IDs are identical with and without the OOV fallback")

rescued = sum(t not in word_to_id and lemmatize_token(t) in word_to_id
              for row in val_records for t in row['tokens'])
print("Unseen validation tokens rescued by lemma fallback:", rescued)

base_model, base_history, base_results = run_experiment(
    embedding_matrix,
    make_loader(train_records, word_to_id, shuffle=True),
    {'Without fallback': make_loader(val_records, word_to_id, use_lemma=False),
     'With fallback': make_loader(val_records, word_to_id, use_lemma=True)},
)
all_results.update(base_results)

In [ ]:
full_train_records = lemmatize_records(train_records)
full_val_records = lemmatize_records(val_records)
full_word_to_id = build_vocab(full_train_records)

print("Full-lemma vocabulary size:", len(full_word_to_id), "| original:", len(word_to_id))
print("Example:", train_records[0]['tokens'], "->", full_train_records[0]['tokens'])

full_model, full_history, full_results = run_experiment(
    build_embedding_matrix(full_word_to_id),
    make_loader(full_train_records, full_word_to_id, shuffle=True),
    {'Full lemmatization': make_loader(full_val_records, full_word_to_id)},
)
all_results.update(full_results)

In [ ]:
comparison = pd.DataFrame.from_dict(all_results, orient='index')
comparison.index.name = 'Variant'
print(comparison.to_string())